# Geant4 Simulation

The Geant4 simulation step is essential for accurately modeling the passage of particles through matter. It simulates the interactions of particles with the detector materials, allowing us to understand how particles lose energy, scatter, and produce secondary particles. This step is crucial for obtaining realistic hit information that will be used in subsequent analysis stages.

To run the Geant4 simulation, we typically use the `simu_trf.py` script, which takes the output from the event generation step as input. The script can be configured to enable various features, such as magnetic fields, which are important for simulating charged particle trajectories accurately.

In [1]:
!simu_trf.py

Module libc not found.
Usage: simu_trf.py -i INPUT_FILE -o OUTPUT_FILE [--nov NUMBER_OF_EVENTS]
                   [-nt NUMBER_OF_THREADS] [--enable-magnetic-field]
                   [-t TIMEOUT] [-l OUTPUT_LEVEL] [--pre-init PRE_INIT]
                   [--pre-exec PRE_EXEC] [--post-exec POST_EXEC]
                   [--save-all-hits] [--dry-run] [--job-file JOB_FILE]

Options:
  -i, --input-file INPUT_FILE
                        The input file or folder to run the job
  -o, --output-file OUTPUT_FILE
                        The output file.
  --nov, --number-of-events NUMBER_OF_EVENTS
                        The total number of events to run.
  -nt, --number-of-threads NUMBER_OF_THREADS
                        The number of threads
  --enable-magnetic-field
                        Enable the magnetic field.
  -t, --timeout TIMEOUT
                        Event timeout in minutes
  -l, --output-level OUTPUT_LEVEL
                        The output level messenger.
  --pre-init PRE_IN

In [2]:
!simu_trf.py -i Zee.EVT.root -o Zee.HIT.root --nov 10 -nt 10 --dry-run

Module libc not found.
input file: Zee.EVT.root
output file: Zee.HIT.root
number of threads: 10
Cxx.ComponentAccumulator                 INFO Run manager was created.
Compiling...: 100%|█████████████████| 34/34 [00:00<00:00, 3267.04it/s]
Py.CaloHitBuilder                       INFO Configure CaloHitBuilder.
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_PSB collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB2 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB3 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_TileCal1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into C

## Magnetic field

By default the simulation runs without magnetic field. Two options enable a 2 T axial field:

* `--enable-solenoid-field`: the field exists only inside the volume of the ATLAS central solenoid, a cylinder with radius 1.23 m and half-length 2.9 m (JINST 3 (2008) S08003, table 2.1). Particles are bent on their way to the calorimeters, and the showers inside the calorimeters are not. The solenoid coil and cryostat material are not modelled, and the field is uniform (in ATLAS it drops from about 1.8 T at $z$ = 1.7 m to about 0.9 T at the end of the inner detector cavity).
* `--enable-magnetic-field`: a uniform 2 T field in the whole world, including the calorimeters.

The two options cannot be used together. With `--enable-solenoid-field`, soft charged particles curl up inside the solenoid instead of being removed at generation, so the events should be generated without the pT cut, as ATLAS does:

```
gen_minbias.py ... --pt-min-charged 0 --pt-min-neutral 0
simu_trf.py -i MB.EVT.root -o MB.HIT.root --enable-solenoid-field
```

## ATLAS-like tile calorimeter

By default the tile calorimeter is made of cylindrical shells of steel and scintillator stacked in radius. The option `--tile-atlas-geometry` builds it as in ATLAS instead:

* the scintillating tiles are placed radially and normal to the beam line, stacked along $z$ in periods of 18 mm: 14 mm of steel (5 mm master plate, 4 mm spacer plate, 5 mm master plate), a 3 mm tile and 1 mm left by the tile in its 4 mm pocket, so that steel:scintillator is 4.67:1 (TILECAL, hep-ex/9904032; JINST 3 (2008) S08003, sec. 5.3.1.2);
* the layer radii are those of the ATLAS cells: 2300, 2600, 3440 and 3820 mm in the barrel (A, BC, D) and 2300, 2600, 3140 and 3820 mm in the extended barrel (A, B, D). With this period they give 1.45/4.07/1.84 interaction lengths in the barrel and 1.45/2.62/3.30 in the extended barrel, against about 1.5/4.1/1.8 and 1.5/2.6/3.3 quoted by ATLAS;
* the $z$ extent is that of the ATLAS cells (JINST 3 (2008) S08003, fig. 5.12), rounded to whole periods: long barrel $|z|$ < 2808 mm (312 periods), extended barrel 3554 < $|z|$ < 6110 mm (142 periods). The aluminium block that stands for the ITC fills the space in between (2808 to 3554 mm), and the aluminium shell inside the barrel follows its length.

Not modelled: the staggering of the tiles between radial rows, the support girder at the outer radius, and the gap and crack scintillators (C10, D4, E1-E4). Without `--tile-atlas-cells` the cells keep the default $\eta \times \phi$ segmentation ($\Delta\eta$ = 0.1, 0.1, 0.2; 64 bins in $\phi$), whose $\eta$ edges follow the new volumes. Since the cells depend on the geometry, the same option must be given to the digitization:

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --tile-atlas-geometry
digit_trf.py -i MB.HIT.root -o MB.ESD.root --tile-atlas-geometry
```

### ATLAS tile cells

The default $\eta \times \phi$ grid is not the ATLAS cell structure, and it leaves part of the tile volume outside any cell: the grid stops at the last full $\eta$ bin, and the energy deposited beyond it is lost (up to 20% of the area of the D layer in the $r$-$z$ plane). With `--tile-atlas-cells` (only together with `--tile-atlas-geometry`) the cells are those of ATLAS, where a cell is the group of tiles read out by the same photomultiplier, with fixed $z$ boundaries in each radial row (JINST 3 (2008) S08003, sec. 5.3.1.4 and fig. 5.12):

* long barrel: A1-A10, BC1-BC8 (the B and C rows read together, with the B boundaries below $r$ = 2990 mm and the C boundaries above), B9 (B row only) and D0-D3, where D0 spans $z$ = 0;
* extended barrel: A12-A16, B11-B15, D5 and D6;
* every boundary is moved to the nearest boundary between two 18 mm periods, so that each tile belongs to a single cell;
* the cell is found from the $(r, z)$ position of each Geant4 step, so that every point of the tile volume belongs to a cell; $\phi$ keeps the 64 slices of $\Delta\phi = \pi/32$;
* each cell is stored with the $\eta$ of its geometric centre and the $\Delta\eta$ at its mean radius; per module there are 45 cells in the barrel and 12 in each extended barrel, 4416 cells in total. The table of cells, with the ATLAS names, is built by `getAtlasTileCells` in `geometry/python/v1/TILE.py`.

The digitization does not support these cells yet: `digit_trf.py --tile-atlas-cells` stops with an error, and the hits must not be digitized with the default cells.

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --tile-atlas-geometry --tile-atlas-cells
```

### Dual readout of the tile cells

In ATLAS each tile cell is read by two photomultipliers, one on each side of the tiles in $\phi$, through the wavelength-shifting fibres that run along the two edges of the tiles. The ATLAS simulation shares the energy of each Geant4 step between the two PMTs according to its azimuthal position across the module (Athena, `TileGeoG4SDCalc::MakePmtEdepTime`). With `--tile-dual-readout` (only together with `--tile-atlas-cells`) the same is done here, and each tile cell stores, besides `edep`, the energy seen by each PMT per bunch crossing (`edep_pmt0`, `edep_pmt1`). The sharing is chosen with `--tile-pmt-split`:

* `ushape` (default): the measured response of each PMT across the tile, the default of the ATLAS simulation (`Tile_1D_profileRescaled`, 99 bins over $\pm$0.0495 rad, one table per layer for the long barrel and for each extended barrel, copied from Athena). The two PMTs do not add up exactly to the cell energy, as in ATLAS;
* `linear`: $0.5 \pm 0.2$ from the centre to the edges of the module, the two PMTs adding up to the cell energy.

The PMT 1 is the one on the side of larger $\phi$, on both sides of the detector (a convention: the orientation of the ATLAS local axes was not checked). D0, which spans $z$ = 0, is one cell read by two PMTs like the others; which of them goes to each side of the detector is a matter of cabling and is not modelled. Both PMTs keep the time of the cell. The digitization and the pile-up merging do not use the PMT energies yet.

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --tile-atlas-geometry --tile-atlas-cells --tile-dual-readout
```

## Material in front of the barrel electromagnetic calorimeter

By default only 40 mm of aluminium stand in front of the barrel presampler, about 0.45 radiation lengths at normal incidence, where ATLAS has about 1.9 $X_0$ in front of the presampler and 2.5 $X_0$ in front of the accordion (JINST 3 (2008) S08003, fig. 5.1). With `--atlas-material-in-front` the material outside the inner detector follows ATLAS, at normal incidence:

* the solenoid, 0.66 $X_0$ (JINST 3 S08003, sec. 2.1.1), in the 50 mm of the coil (inner and outer diameters 2.46 and 2.56 m, length 5.8 m): 1.68 mm of copper and 48.32 mm of aluminium, just outside the volume of `--enable-solenoid-field`;
* the cryostat wall in front of the presampler, 0.76 $X_0$ = 67.6 mm of aluminium in the same 80 mm envelope as the default 40 mm: the 1.86 $X_0$ of ATLAS in front of the presampler less the inner detector (0.44 $X_0$, taken from the Run 2 inner detector, ATLAS-TDR-030, fig. 2.6) and the solenoid;
* the material between the presampler and the accordion, 0.60 $X_0$ (0.68 $X_0$ in ATLAS less the presampler itself): 4.71 mm of copper and 24.19 mm of aluminium in the 29 mm between them.

Not modelled: the inner detector, the tapered walls of the barrel cryostat (the cylinders give up to about 10% more $X_0$ than ATLAS above $|\eta| \approx 1$), the end-cap cryostat walls and the services. The option changes the simulation only; the cells and the digitization are not affected.

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --enable-solenoid-field --atlas-material-in-front
```

### The gap between the tile barrel and the extended barrel

With `--tile-atlas-geometry` the gap between the tile barrel ($|z|$ < 2808 mm) and the extended barrel ($|z|$ > 3554 mm) is filled by a block of aluminium 2 m thick in radius, which stands for the ITC. In ATLAS the gap "is filled with cables and services", with "a reduced section of a standard tile-calorimeter sub-module, the plug" at the outer radius (JINST 3 (2008) S08003, sec. 5.5). With `--tile-atlas-itc` (only together with `--tile-atlas-geometry`) the block is replaced by:

* the plug: the D4 (17 periods, $z$ = 3227 to 3533 mm, $r$ = 3461 to 3850 mm) and C10 (4 periods, $z$ = 3451.5 to 3523.5 mm, $r$ = 2988 to 3457 mm) parts, boxes from fig. 5.12 of the same paper, with tiles normal to the beam line and the period of the ATLAS-like tile calorimeter; they are passive (no cells);
* the cables and services in the rest of the gap, as aluminium with 5% of the volume in shells of 50 mm or less. The paper does not give their mass: the fraction comes from a fit of the interaction lengths at the end of the tile calorimeter to fig. 5.2 of the paper for $|\eta|$ = 0.85 to 1.05, which gives 0 to 5%.

Not modelled: the gap and cryostat scintillators E1-E4. The option changes the simulation only.

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --tile-atlas-geometry --tile-atlas-cells --tile-atlas-itc
```

## ATLAS-like barrel electromagnetic calorimeter

By default the three layers of the barrel EM calorimeter are cylindrical shells of 1.51 or 1.7 mm of lead and liquid argon in periods of 6 mm, from $r$ = 1500 to 1980 mm. In ATLAS the absorbers are lead plates of 1.53 mm for $|\eta|$ < 0.8 and 1.13 mm for $|\eta|$ > 0.8 (the change limits the decrease of the sampling fraction), with two 0.2 mm stainless-steel sheets glued on them, and the drift gap is 2.1 mm on each side of the readout electrode (JINST 3 (2008) S08003, sec. 5.2). With `--atlas-emb`:

* each absorber is a homogeneous mixture with the masses per unit area of the lead, of the two steel sheets (as iron) and of 0.832 mm of polyimide for the glue and the electrode, whose thickness is not given in the paper and is fitted to the 22.4 $X_0$ of the accordion at $\eta$ = 0 (fig. 5.1 of the paper);
* the layers keep their division (14, 47 and 7 periods of 6.962 mm) and end at 1973.4 mm, 470 mm of depth as in fig. 5.4 of the paper; the $X_0$ of the three layers together follows fig. 5.1 within about 2% up to $|\eta|$ = 1.2;
* the lead changes from 1.53 to 1.13 mm at a cut in $z$ at the mean radius of each layer, $z = r \sinh 0.8$ (about $\pm$0.04 in $\eta$ at the inner and outer radius); with thinner lead the argon takes the room, since the number of absorbers and the folds are fixed.

Not modelled: the accordion shape and the depth segmentation that follows $\eta$ in ATLAS (the $X_0$ of each layer differs, the total agrees). The cells depend on the layer radii, so the same option must be given to the digitization:

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --atlas-emb
digit_trf.py -i MB.HIT.root -o MB.ESD.root --atlas-emb
```

## End-cap cryostat walls

By default nothing stands between the outer radius of the EMEC and HEC (2042 and 2030 mm) and the inner radius of the extended tile barrel (2283 mm, or 2300 mm with `--tile-atlas-geometry`). In ATLAS each end-cap cryostat is made of two aluminium vessels, a cold vessel filled with liquid argon inside a warm vessel, and their outer cylinders lie there (JINST 3 (2008) S08003, sec. 5.4 and fig. 5.25: outer radius 2.25 m, length 3.17 m). With `--atlas-endcap-cryostat`, on both sides:

* the outer cylinder of the warm vessel, 20 mm of aluminium at $r$ = 2260 to 2280 mm, up to 2630 mm behind its front face;
* the outer cylinder of the cold vessel, 35 mm of aluminium at $r$ = 2140 to 2175 mm, 3015 mm long;
* the liquid argon that fills the cold vessel between the EMEC and the cold cylinder, as dead material (no cells), up to the 88 mm back wall.

The thicknesses and lengths are those of the LAr calorimeter TDR (CERN/LHCC 96-41, fig. 3-3 and 5-i), with the front face of the warm vessel at $z$ = 3500 mm moved 40 mm away from the interaction point as in the installed detector (JINST 3 (2008) S08003, sec. 5.2). The cylinders start at $z$ = 3717.5 mm, the end of the aluminium that already stands for the front walls (`DM::Crack::EM`). A particle that reaches the A layer of the extended barrel at $|\eta|$ = 1.35 to 1.55 crosses 2.7 to 3.3 $X_0$ and 0.5 to 0.6 interaction lengths of this material.

Not modelled: the step of the warm vessel to a diameter of 4950 mm in its last 535 mm (behind the end of the tile calorimeter), the thicker back part of the cold cylinder, the feed-throughs and the cables. The option changes the simulation only.

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --atlas-endcap-cryostat
```

## Barrel cryostat

By default two aluminium shells of 100 mm stand around the barrel calorimeters: one just outside the EM barrel ($r$ = 1980 to 2080 mm, $|z|$ < 3400 mm) and one inside the tile barrel ($r$ = 2180 to 2280 mm). In ATLAS the barrel cryostat is made of two aluminium vessels, a cold vessel filled with liquid argon inside a warm vessel (JINST 3 (2008) S08003, sec. 5.4), with thinner outer walls and its ends in the gap between the tile barrel and the extended barrel. With `--atlas-barrel-cryostat` (only together with `--tile-atlas-itc`) the two shells are replaced by the outer part of the ATLAS barrel cryostat, from the LAr calorimeter TDR (CERN/LHCC 96-41, fig. 3-2):

* the outer cylinder of the cold vessel, 30 mm of aluminium at $r$ = 2140 to 2170 mm, with end walls of 30 mm at $|z|$ = 3267 to 3297 mm and, at each end, a 220 mm step up to $r$ = 2288.5 mm for the feed-throughs;
* the outer cylinder of the warm vessel, 30 mm at $r$ = 2220 to 2250 mm, with end walls of 50 mm at $|z|$ = 3355 to 3405 mm and, at each end, a 540 mm step up to $r$ = 2775 mm;
* the liquid argon that fills the cold vessel between the EM barrel and the cold cylinder, as dead material (no cells).

The walls of the steps are not dimensioned in the TDR and are taken as 30 mm like the other walls. The EM barrel is 250 mm longer than in ATLAS ($|z|$ < 3400 against 3150 mm), so the end walls only cover the radii outside it ($r$ > 1980 mm). The cables and services of the gap (`--tile-atlas-itc`) leave room for the step of the warm vessel and keep their 5% of aluminium in the remaining volume. Not modelled: the feed-throughs, the inner walls of the cryostat (see `--atlas-material-in-front` for the material in front of the EM barrel). The option changes the simulation only.

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --tile-atlas-geometry --tile-atlas-cells --tile-atlas-itc --atlas-barrel-cryostat
```

## Free-running hits in the layout of the ATLAS HITS ntuple

The standard HIT stream keeps one energy per cell and bunch crossing, in a fixed window. Studies of the free-running readout (continuous sequences of bunch crossings with pile-up) need instead the individual deposits with their time, as in the ATLAS HITS. With `--free-running-hits` (only together with `--tile-atlas-geometry` and `--tile-atlas-cells`) `simu_trf.py` writes, instead of the HIT stream, a tree `CollectionTree` with the branches of the flat ATLAS HITS ntuple: `EventNumber` and `TileCalHit_cellID`, `_energy`, `_time`, `_eta`, `_phi`, `_sampling`, `_side`, `_module`, `_tower`, plus the extra `TileCalHit_tglobal`; the LAr branches (`LArHitEMB_*`, `LArHitEMEC_*`, `LArHitHEC_*`, `LArHitFCAL_*`) are written empty, since the LAr cells of Lorenzetti are not those of ATLAS. As in the ATLAS simulation (Athena, `TileGeoG4SDCalc`, `TileSimHit`; public code):

* one entry per PMT: the option switches on the dual readout with the U-shape sharing, the active medium and Birks' law;
* `cellID` is the ATLAS PMT identifier (`pmt_id`), packed in 64 bits as the ATLAS identifier dictionary does (`IdDictTileCalorimeter.xml`): Tile in bits 63-61, section 60-58, side 57-54, module 53-46, tower 45-40, sampling 39-36, pmt 35-34; module m covers $\phi$ from $m$ to $m+1$ times $2\pi/64$;
* `time` is the global time at the end of the step with the time-of-flight and fibre corrections (refraction index 1.59) and the delay of each PMT; deposits after 350.5 ns are kept with the time 99995 ns;
* the deposits of a PMT are summed in time bins of 0.5 ns between -75.25 and 75.25 ns and of 5 ns outside (`--free-running-no-binning` keeps one entry per step); `tglobal` is the energy weighted mean of the Geant4 global times of the deposits of an entry.

`--free-running-keep-hits` keeps also the standard HIT stream (tree `CollectionTree`) and writes the free-running hits in the tree `FreeRunningTree`, for checks.

```
simu_trf.py -i MB.EVT.root -o MB.FRHITS.root --tile-atlas-geometry --tile-atlas-cells --free-running-hits
```

### Neutron time cut

The physics list (FTFP_BERT) kills every neutron 10 $\mu$s after the start of the event, the default of Geant4. The ATLAS simulation kills them 150 ns after the primary interaction ("a Geant4 neutron time cut is applied which removes all neutrons 150 ns after the primary interaction", ATLAS simulation paper, arXiv:1005.4568, sec. 5.5; `Sim.NeutronTimeCut` in Athena). With `--neutron-time-cut` (in ns) the limit is changed; `--free-running-hits` sets 150 ns unless the option is given. In minimum bias, with 10 $\mu$s about a quarter to a third of the visible energy of the tile calorimeter comes after 350 ns (slow neutrons); with 150 ns about 1%, as in ATLAS. The energy before 150 ns does not change.

```
simu_trf.py -i MB.EVT.root -o MB.HIT.root --neutron-time-cut 150
```

## Pre-execution adaptation:

It's possible to configure certain algorithms within the simulation using the `pre-exec` parameter. Usually this stage can be used when all algorithms are setted. This will be executed before the execution (`run`) event loop.

### Save all hits:

Here is hot can set the simulator to keep all hits.

In [3]:
!simu_trf.py -i Zee.EVT.root -o Zee.HIT.root --nov 10 -nt 10 --dry-run --pre-exec "HIT.setProperty('OnlyRoI',False)"

Module libc not found.
input file: Zee.EVT.root
output file: Zee.HIT.root
number of threads: 10
Cxx.ComponentAccumulator                 INFO Run manager was created.
Compiling...: 100%|█████████████████| 34/34 [00:00<00:00, 3428.94it/s]
Py.CaloHitBuilder                       INFO Configure CaloHitBuilder.
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_PSB collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB2 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB3 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_TileCal1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into C

### Anomaly simulation:

Here is how you can set the KeepCells parameter for the `RootStreamHitMaker` algorithm:
* Parameter: `KeepCells`
* Value: A list of cells specified by their hash number.

**NOTE**: This specific strategy is often utilized in anomaly simulations.

In [4]:
import json

cells = [
                        # examples of cell hash IDs where noise is increased
                             1265,      1266,      1267,      1268,      1269, # center PSB
                        170001265, 170001266, 170001267, 170001268, 170001269, # right PSB
                        340001265, 340001266, 340001267, 340001268, 340001269, # left PSB
                         18400805,  18400806,  18400807,  18400808,  18400809, # center PSE
                        188400805, 188400806, 188400807, 188400808, 188400809, # right PSE
                        358000805, 358000806, 358000807, 358000808, 358000809, # left PSE
                         20025610,  20025611,  20025612,  20025613,  20025614, # center EMB1
                        190025610, 190025611, 190025612, 190025613, 190025614, # right EMB1
                        360025610, 360025611, 360025612, 360025613, 360025614, # left EMB1
                         30029160,  30029161,  30029162,  30029163,  30029164, # center EMB2
                        200029160, 200029161, 200029162, 200029163, 200029164, # right EMB2
                        370029160, 370029161, 370029162, 370029163, 370029164, # left EMB2
                         40013020,  40013021,  40013022,  40013023,  40013024, # center EMB3
                        210013020, 210013021, 210013022, 210013023, 210013024, # right EMB3
                        380013020, 380013021, 380013022, 380013023, 380013024, # left EMB3
                         50000245,  50000246,  50000247,  50000248,  50000249, # center TileCal1
                        220000245, 220000246, 220000247, 220000248, 220000249, # right TileCal1
                        390000245, 390000246, 390000247, 390000248, 390000249, # left TileCal1
                         60000245,  60000246,  60000247,  60000248,  60000249, # center TileCal2
                        230000245, 230000246, 230000247, 230000248, 230000249, # right TileCal2
                        400000245, 400000246, 400000247, 400000248, 400000249, # left TileCal2
                         78406245,  78406246,  78406247,  78406248,  78406249, # center TileCal3
                        248406245, 248406246, 248406247, 248406248, 248406249, # right TileCal3
                        418406245, 418406246, 418406247, 418406248, 418406249, # left TileCal3
                        256000680, 256000681, 256000682, 256000683, 256000684, # right TileExt1
                        426000680, 426000681, 426000682, 426000683, 426000684, # left TileExt1
                        260000120, 260000121, 260000122, 260000123, 260000124, # right TileExt2
                        430000120, 430000121, 430000122, 430000123, 430000124, # left TileExt2
                        270000106, 270000107, 270000108, 270000109, 270000110, # right TileExt3
                        440000106, 440000107, 440000108, 440000109, 440000110, # left TileExt3
                        280000440, 280000441, 280000442, 280000443, 280000444, # right EMEC1
                        450000440, 450000441, 450000442, 450000443, 450000444, # left EMEC1
                        290000440, 290000441, 290000442, 290000443, 290000444, # right EMEC2
                        460000440, 460000441, 460000442, 460000443, 460000444, # left EMEC2
                        300005110, 300005111, 300005112, 300005113, 300005114, # right EMEC3
                        470005110, 470005111, 470005112, 470005113, 470005114, # left EMEC3
                        312807780, 312807781, 312807782, 312807783, 312807784, # right HEC1
                        482807780, 482807781, 482807782, 482807783, 482807784, # left HEC1                        
                        320000560, 320000561, 320000562, 320000563, 320000564, # right HEC2
                        490000560, 490000561, 490000562, 490000563, 490000564, # left HEC2
                        330000290, 330000291, 330000292, 330000293, 330000294, # right HEC3
                        500000290, 500000291, 500000292, 500000293, 500000294, # left HEC3
      ]

d = {
    "runs": [
        {
            "DeadModules"     : True,
            "StartEventNumber": 80000,
            "EndEventNumber"  : 81000,
            "NoiseStdFactor"  : 2,
            "Cells"           : [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
        },
        {
            "DeadModules"     : True,
            "StartEventNumber": 85000,
            "EndEventNumber"  : 87000,
            "NoiseStdFactor"  : 2,
            "Cells"           : [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
        },
        {
            "DeadModules"     : False,
            "StartEventNumber": 89000,
            "EndEventNumber"  : 92000,
            "NoiseStdFactor"  : 2,
            "Cells"           : cells,    
        }
    ] 
}
with open("brl.json", 'w') as f:
    json.dump(d,f)



In [5]:
!simu_trf.py -i Zee.EVT.root -o Zee.HIT.root --nov 1 --pre-exec "from CaloCellBuilder import get_cells_from_brl as reader;HIT.setProperty('KeepCells',reader('brl.json'))"

Module libc not found.
input file: Zee.EVT.root
output file: Zee.HIT.root
number of threads: 1
Cxx.ComponentAccumulator                 INFO Run manager was created.
Compiling...: 100%|█████████████████| 34/34 [00:00<00:00, 3278.76it/s]
Py.CaloHitBuilder                       INFO Configure CaloHitBuilder.
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_PSB collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB2 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB3 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_TileCal1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Co

In [6]:
!ls

Simulation.ipynb  Zee.ESD.root	Zee.HIT.root  dead_modules.json
Untitled.ipynb	  Zee.EVT.root	brl.json
